### 로봇 bring up

In [1]:
# 터미널에서 
# ros2 launch dsr_bringup2 dsr_bringup2_rviz.launch.py mode:=real host:=110.120.1.68 model:=e0509
# ros2 run dsr_gripper gripper_service
# ros2 service call /dsr01/gripper/cmd dsr_gripper_interfaces/srv/GripperCmd "{position: 0, current: 300}"
# ros2 service call /dsr01/gripper/cmd dsr_gripper_interfaces/srv/GripperCmd "{position: 300, current: 0}"

### 로봇 연결

In [1]:
import rclpy
# --- 로봇 설정 ---
ROBOT_ID = "dsr01" # namespace로 사용됨 (현재 시스템: /dsr01/...)
ROBOT_MODEL = "e0509"
import DR_init
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL
rclpy.init()
# ★ 핵심: namespace=dsr01 로 노드 생성 -> 서비스가 /dsr01/dsr_controller2/... 로 연결됨
node = rclpy.create_node("doosan_notebook", namespace=ROBOT_ID)
DR_init.__dsr__node = node
# 이제 API import (위 순서 지켜야 함)
from DSR_ROBOT2 import (
movej, movel, move_home,
get_current_posj, get_current_posx,
set_robot_mode, get_robot_mode, set_velj, set_accj, wait,
task_compliance_ctrl, release_compliance_ctrl, set_ref_coord,
set_tool_digital_output,
ROBOT_MODE_AUTONOMOUS, ROBOT_MODE_MANUAL,
DR_BASE, DR_TOOL, DR_MV_MOD_ABS, DR_MV_MOD_REL, DR_HOME_TARGET_MECHANIC,
)
from DSR_ROBOT2 import posj, posx
# 자동(autonomous) 모드로 전환
set_robot_mode(ROBOT_MODE_AUTONOMOUS)
print("로봇이 연결되었습니다.")

_robot_id =dsr01
_robot_model =e0509
_srv_name_prefix =dsr_controller2/
_topic_name_prefix =dsr_controller2/
로봇이 연결되었습니다.


### 로봇의 현재 데이터 읽기

In [2]:
# 현재 관절 각도
j = get_current_posj()
print("현재 관절 각도(deg):", j)
# 현재 TCP 좌표 (posx, sol_space) 튜플 반환
x, sol = get_current_posx()
print("현재 좌표 [x,y,z,rx,ry,rz]:", x)
print("solution space:", sol)

현재 관절 각도(deg): [-0.000, -0.000, 90.000, -0.000, 90.000, 0.000]
현재 좌표 [x,y,z,rx,ry,rz]: [373.000, -0.000, 405.000, 172.349, -180.000, 172.349]
solution space: 2


### 로봇을 초기 위치로 이동

In [3]:
home = posj(0, 0, 0, 0, 0, 0)
print("로봇을 초기 위치로 이동합니다.")
movej(home, vel=20, acc=20)
print("완료")

로봇을 초기 위치로 이동합니다.
완료


### 단일 관절 움직이기

In [4]:
# J1을 +30도 상대이동
print("J1을 +30도 회전")
movej(posj(30, 0, 0, 0, 0, 0), vel=20, acc=20, mod=DR_MV_MOD_REL)
# J1을 -30도 상대이동 (원위치)
print("J1을 -30도 복귀")
movej(posj(-30, 0, 0, 0, 0, 0), vel=20, acc=20, mod=DR_MV_MOD_REL)
print("완료")

J1을 +30도 회전
J1을 -30도 복귀
완료


### 모든 관절 각도 움직이기

In [5]:
target = posj(20, 20, -20, 20, 20, -45)
print(f"모든 관절을 {list(target)}로 이동")
movej(target, vel=20, acc=20)
print("초기 위치로 복귀")
movej(posj(0, 0, 0, 0, 0, 0), vel=20, acc=20)
print("완료")

모든 관절을 [20, 20, -20, 20, 20, -45]로 이동
초기 위치로 복귀
완료


### 좌표(Task)로 이동 (movel)

In [ ]:
# 1) 특이점 회피: 먼저 movej로 준비자세로 이동
ready = posj(0, 0, 90, 0, 90, 0)
print("준비자세로 이동 (movej)")
movej(ready, vel=30, acc=30)
# 2) 현재 좌표 확인
cur, _ = get_current_posx()
print("현재 좌표:", cur)
# 3) movel 직선이동 (상대이동 REL — 현재 위치 기준)
print("Z -50mm 내리기")
movel(posx(0, 0, -50, 0, 0, 0), vel=30, acc=30, ref=DR_BASE, mod=DR_MV_MOD_REL)
print("X +50mm")
movel(posx(50, 0, 0, 0, 0, 0), vel=30, acc=30, ref=DR_BASE, mod=DR_MV_MOD_REL)
print("Y +50mm")
movel(posx(0, 50, 0, 0, 0, 0), vel=30, acc=30, ref=DR_BASE, mod=DR_MV_MOD_REL)
# 4) 최종 좌표 확인 후 준비자세로 복귀
final, _ = get_current_posx()
print("최종 좌표:", final)
print("준비자세로 복귀")
movej(ready, vel=30, acc=30)
print("완료")

### 상대좌표 이동 (한 번에)

In [ ]:
# 준비자세로 이동 후 상대좌표 한 번에 이동 (X+50, Y-50, Z-50)
ready = posj(0, 0, 90, 0, 90, 0)
movej(ready, vel=30, acc=30)
delta = posx(50, -50, -50, 0, 0, 0)
print("상대좌표로 이동:", list(delta))
movel(delta, vel=30, acc=30, ref=DR_BASE, mod=DR_MV_MOD_REL)
print("준비자세로 복귀")
movej(ready, vel=30, acc=30)
print("완료")

### 비동기(async) 이동 제어

In [28]:
# 비동기 이동: amovej / amovel 은 명령을 내리고 '즉시' 리턴합니다.
#   - movej / movel   : 이동이 끝날 때까지 블로킹(대기)
#   - amovej / amovel : 즉시 리턴 -> 이동 중에 다른 일을 할 수 있음
#   - check_motion()  : 이동 상태 폴링 (0 = 이동 없음/완료)
#   - mwait()         : 완료 대기 한 줄 버전 (단, 빌드에 따라 즉시 반환될 수 있음)
from DSR_ROBOT2 import amovej, amovel, mwait, check_motion
import time
# 8번과 같은 준비자세를 기준으로 잡고, 여기서 가깝게 상대이동합니다.
ready = posj(0, 0, 90, 0, 90, 0)
movej(ready, vel=20, acc=20)
# --- 방법 1) amovej (관절 비동기) + check_motion() 폴링 ---
print("amovej: 준비자세에서 J1 +20도 (즉시 리턴)")
amovej(posj(20, 0, 0, 0, 0, 0), vel=20, acc=20, mod=DR_MV_MOD_REL)
while check_motion() != 0:
    j = get_current_posj()
    print("  이동 중... J1 =", round(j[0], 1))
    time.sleep(0.3)
print("이동 완료")
movej(ready, vel=20, acc=20)              # 준비자세로 복귀
# --- 방법 2) amovel (직선 비동기) + check_motion() 완료 대기 ---
# mwait() 한 줄로도 되지만, 일부 빌드에서 즉시 반환되어 바로 아래 복귀 이동과
# 겹쳐(blend) 움직이지 않는 것처럼 보일 수 있어, 방법1과 같은 check_motion() 폴링을 씁니다.
print("amovel: 현재 위치에서 Z -50mm 직선 (즉시 리턴)")
amovel(posx(0, 0, -50, 0, 0, 0), vel=20, acc=20, ref=DR_BASE, mod=DR_MV_MOD_REL)
while check_motion() != 0:
    x, _ = get_current_posx()
    print("  이동 중... Z =", round(x[2], 1))
    time.sleep(0.3)
print("이동 완료")
movel(posx(0, 0, 50, 0, 0, 0), vel=20, acc=20, ref=DR_BASE, mod=DR_MV_MOD_REL)  # 복귀
print("복귀 완료")

amovej: 준비자세에서 J1 +20도 (즉시 리턴)
  이동 중... J1 = 0.0
  이동 중... J1 = 0.2
  이동 중... J1 = 2.2
  이동 중... J1 = 6.9
  이동 중... J1 = 12.7
  이동 중... J1 = 17.5
  이동 중... J1 = 19.7
이동 완료
amovel: 현재 위치에서 Z -50mm 직선 (즉시 리턴)
  이동 중... Z = 405.0
  이동 중... Z = 404.7
  이동 중... Z = 401.9
  이동 중... Z = 396.6
  이동 중... Z = 390.4
  이동 중... Z = 384.3
  이동 중... Z = 378.1
  이동 중... Z = 372.0
  이동 중... Z = 365.8
  이동 중... Z = 360.1
  이동 중... Z = 356.2
  이동 중... Z = 355.0
이동 완료
복귀 완료


### 동기 vs 비동기 비교 (직선 경로 6점)

In [29]:
# 비교 준비(셋업): 라이브러리 + 직선 경로 6점(웨이포인트) 정의
#   * A/B 셀을 실행하기 전에 이 셀을 먼저 한 번 실행하세요.
import time
from DSR_ROBOT2 import amovel, movel, check_motion
ready = posj(0, 0, 90, 0, 90, 0)
movej(ready, vel=20, acc=20)
p0, _ = get_current_posx()
def wp(dx, dy, dz):
    q = list(p0); q[0] += dx; q[1] += dy; q[2] += dz
    return posx(q)
# X 방향으로 20mm씩 직진하는 6개 좌표
waypoints = [wp(20, 0, 0), wp(40, 0, 0), wp(60, 0, 0),
             wp(80, 0, 0), wp(100, 0, 0), wp(120, 0, 0)]
print("웨이포인트 준비 완료:", len(waypoints), "점")

웨이포인트 준비 완료: 6 점


In [30]:
# ---------- (A) 그냥 제어 (동기 movel) ----------
# movel 은 '도착할 때까지' 코드가 멈춤(블로킹). 이동 중엔 아무 것도 못 하고, 도착 후에만 출력.
movej(ready, vel=20, acc=20)
print("[동기] 시작")
t0 = time.time()
for i, p in enumerate(waypoints):
    movel(p, vel=40, acc=40, ref=DR_BASE)       # 도착할 때까지 블로킹
    print(f" waypoint {i+1} 도착 (도착 후에야 이 줄이 실행됨)")
print("[동기] 총 %.2f초" % (time.time() - t0))

[동기] 시작
 waypoint 1 도착 (도착 후에야 이 줄이 실행됨)
 waypoint 2 도착 (도착 후에야 이 줄이 실행됨)
 waypoint 3 도착 (도착 후에야 이 줄이 실행됨)
 waypoint 4 도착 (도착 후에야 이 줄이 실행됨)
 waypoint 5 도착 (도착 후에야 이 줄이 실행됨)
 waypoint 6 도착 (도착 후에야 이 줄이 실행됨)
[동기] 총 9.24초


In [31]:
# ---------- (B) 비동기 제어 (amovel) ----------
# amovel 은 즉시 리턴 -> check_motion() 대기 없이 6개 명령을 순식간에 다 던집니다.
# 코드는 바로 끝나지만, 로봇은 뒤에서 큐에 쌓인 순서대로 계속 이동합니다.
movej(ready, vel=20, acc=20)
print("[비동기] 시작")
t0 = time.time()
for i, p in enumerate(waypoints):
    amovel(p, vel=40, acc=40, ref=DR_BASE)       # 즉시 리턴 (대기 없음)
    print(f" wp{i+1} 명령 던짐 (즉시 리턴)")
print("[비동기] 총 %.2f초 - 명령만 다 던지고 끝 (로봇은 아직 이동 중)" % (time.time() - t0))

[비동기] 시작
 wp1 명령 던짐 (즉시 리턴)
 wp2 명령 던짐 (즉시 리턴)
 wp3 명령 던짐 (즉시 리턴)
 wp4 명령 던짐 (즉시 리턴)
 wp5 명령 던짐 (즉시 리턴)
 wp6 명령 던짐 (즉시 리턴)
[비동기] 총 4.32초 - 명령만 다 던지고 끝 (로봇은 아직 이동 중)


In [11]:
# ---------- (B) 비동기 제어 (amovel) ----------
# amovel 은 즉시 리턴 -> check_motion() 대기 없이 6개 명령을 순식간에 다 던집니다.
# 코드는 바로 끝나지만, 로봇은 뒤에서 큐에 쌓인 순서대로 계속 이동합니다.
movej(ready, vel=20, acc=20)
print("[비동기] 시작")

[비동기] 시작


### 그리퍼 서비스 노드 빌드 & 실행

In [ ]:
# 터미널에서 선행 해야한다
# cd ~/doosan_ws
# colcon build --packages-select dsr_gripper_interfaces dsr_gripper
# source install/setup.bash
# ros2 run dsr_gripper gripper_service

### 그리퍼 API 불러오기

In [6]:
from dsr_gripper import gripper_open, gripper_close, gripper_cmd
print("그리퍼 API 준비 완료")

그리퍼 API 준비 완료


### 그리퍼 단독 동작

In [15]:
# 그리퍼 열기
gripper_open()

True

In [23]:
# 그리퍼 닫기 (힘 300)
gripper_close(current=300)

True

In [16]:
# 반쯤 열기 (힘 200)
gripper_cmd(375, current=100)

True

### manual(수동) 모드로 전환

In [27]:
# 현재 모드 확인
print("현재 로봇 모드:", get_robot_mode())
# manual(수동) 모드로 전환
set_robot_mode(ROBOT_MODE_MANUAL)
print("전환 후 모드:", get_robot_mode(), "(0이면 manual)")

현재 로봇 모드: 0
전환 후 모드: 0 (0이면 manual)


### 로봇 팔의 직접교시 버튼을 누른 채원하는 자세로 팔을 움직이기

In [ ]:
taught = get_current_posj()
print("티칭한 관절 각도(deg):", taught)

### 티칭이 끝나면 autonomous(자동) 모드로 복귀

In [ ]:
set_robot_mode(ROBOT_MODE_AUTONOMOUS)
print("전환 후 모드:", get_robot_mode(), "(1이면 autonomous)")

### TCP 설정(매뉴얼 모드 필요)

In [ ]:
from DSR_ROBOT2 import add_tcp, set_tcp, get_tcp
# 1) 설정 전 좌표 (플랜지 기준)
before, _ = get_current_posx()
print("TCP 설정 전 좌표(플랜지 기준):", [round(v, 1) for v in before])
# 2) manual 모드로 전환해야 add_tcp/set_tcp 가 동작함
set_robot_mode(ROBOT_MODE_MANUAL)
wait(1)
#    [x, y, z, rx, ry, rz] (mm, deg) — 플랜지에서 툴 끝까지 오프셋
#    ⚠️ Z=150 은 예시. RH-P12-RN 실측 길이로 수정
add_tcp("gripper_tcp", [0, 0, 150, 0, 0, 0])
set_tcp("gripper_tcp")
print("현재 TCP:", get_tcp())
# 3) 다시 auto(autonomous) 모드로 복귀 (movej/movel 하려면 필수)
set_robot_mode(ROBOT_MODE_AUTONOMOUS)
wait(1)
# 4) 설정 후 좌표 (그리퍼 끝점 기준) — 오프셋만큼 달라짐
after, _ = get_current_posx()
print("TCP 설정 후 좌표(그리퍼 끝점 기준):", [round(v, 1) for v in after])

### 노트북 노드 종료

In [ ]:
node.destroy_node()
rclpy.shutdown()
print("종료 완료")

### Bring up 명령 실행한 터미널에서 ROS2 종료 / 서보토크 해제

In [ ]:
# ros2 launch dsr_bringup2 dsr_bringup2_rviz.launch.py mode:=real host:=110.120.1.68 model:=e0509
# 이 명령 실행한 터미널에서 ctrl + c  필요